# 09 — Batch Scoring: 3.9M Registros

Este notebook aplica o modelo champion (ensemble ou melhor individual) sobre toda a **Feature Store** (~3.9M registros) para gerar:

- **Probabilidade de FPD** — saída bruta do modelo
- **Score** — escala 0-1000 (quanto maior, menor o risco)
- **Faixa de Risco** — classificação categórica (CRITICO, ALTO, MEDIO, BAIXO)

Os scores são escritos em formato particionado por SAFRA no bucket Gold.

In [ ]:
import numpy as np
import pandas as pd
import pickle
import uuid
import os
import json
from datetime import datetime
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from config import *
from utils import *

In [ ]:
# ---------------------------------------------------------------------------
# Carregar modelo champion
# ---------------------------------------------------------------------------
champion_path = os.path.join(ARTIFACT_DIR, 'models', 'champion_ensemble.pkl')

if os.path.exists(champion_path):
    with open(champion_path, 'rb') as fh:
        champion_model = pickle.load(fh)
    print(f"Modelo champion (ensemble) carregado: {champion_path}")
else:
    # Fallback: carregar melhor modelo individual
    model_dir = os.path.join(ARTIFACT_DIR, 'models')
    model_files = [f for f in os.listdir(model_dir) if f.endswith('.pkl')]
    fallback_path = os.path.join(model_dir, model_files[0])
    with open(fallback_path, 'rb') as fh:
        champion_model = pickle.load(fh)
    print(f"Modelo individual carregado (fallback): {fallback_path}")

# Carregar metadata
metadata_path = os.path.join(ARTIFACT_DIR, 'champion_metadata.json')
if os.path.exists(metadata_path):
    with open(metadata_path, 'r') as fh:
        champion_metadata = json.load(fh)
    model_version = champion_metadata.get('strategy', 'individual')
else:
    model_version = 'v1.0'

print(f"Versão do modelo: {model_version}")

In [ ]:
# ---------------------------------------------------------------------------
# Carregar Feature Store completa (projeção de colunas)
# ---------------------------------------------------------------------------
fs_path = LOCAL_DATA_PATH
columns_to_load = ['NUM_CPF', 'SAFRA'] + SELECTED_FEATURES

df = pd.read_parquet(fs_path, columns=columns_to_load)
print(f"Feature Store carregada: {df.shape[0]:,} registros x {df.shape[1]} colunas")
print(f"SAFRAs: {sorted(df['SAFRA'].unique())}")

## Scoring

A conversão segue a fórmula:
1. **Probabilidade** — `predict_proba(X)[:, 1]` (prob. classe FPD=1)
2. **Score** — `((1 - prob) * 1000).astype(int).clip(0, 1000)` — escala invertida, score alto = baixo risco
3. **Faixa de Risco** — mapeamento via `risk_band()` (CRITICO, ALTO, MEDIO, BAIXO)

In [ ]:
# ---------------------------------------------------------------------------
# Scoring
# ---------------------------------------------------------------------------
X = df[SELECTED_FEATURES]

probabilities = champion_model.predict_proba(X)[:, 1]
scores = ((1 - probabilities) * 1000).astype(int).clip(0, 1000)
risk_bands = pd.Series(scores).apply(risk_band).values

print(f"Scoring concluído para {len(scores):,} registros")
print(f"Probabilidade — min: {probabilities.min():.4f}, max: {probabilities.max():.4f}, média: {probabilities.mean():.4f}")
print(f"Score — min: {scores.min()}, max: {scores.max()}, média: {scores.mean():.1f}")

In [ ]:
# ---------------------------------------------------------------------------
# Construir DataFrame de scores
# ---------------------------------------------------------------------------
execution_id = str(uuid.uuid4())
data_scoring = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

scores_df = pd.DataFrame({
    'NUM_CPF': df['NUM_CPF'].values,
    'SAFRA': df['SAFRA'].values,
    'SCORE': scores,
    'FAIXA_RISCO': risk_bands,
    'PROBABILIDADE_FPD': probabilities.round(6),
    'MODELO_VERSAO': model_version,
    'DATA_SCORING': data_scoring,
    'EXECUTION_ID': execution_id,
})

print(f"DataFrame de scores: {scores_df.shape}")
print(f"Execution ID: {execution_id}")
scores_df.head(10)

## Distribuicao por Faixa de Risco

Tabela de distribuição das faixas de risco e estatísticas descritivas do score.

In [ ]:
# ---------------------------------------------------------------------------
# Distribuição por faixa de risco
# ---------------------------------------------------------------------------
band_order = ['CRITICO', 'ALTO', 'MEDIO', 'BAIXO']
dist = scores_df['FAIXA_RISCO'].value_counts()
dist = dist.reindex(band_order, fill_value=0)
dist_pct = (dist / len(scores_df) * 100).round(2)

dist_table = pd.DataFrame({
    'Faixa': band_order,
    'Quantidade': dist.values,
    'Percentual (%)': dist_pct.values,
})

print("=== Distribuição por Faixa de Risco ===")
print(dist_table.to_string(index=False))

print(f"\n=== Estatísticas do Score ===")
print(f"  Média:   {scores_df['SCORE'].mean():.1f}")
print(f"  Mediana: {scores_df['SCORE'].median():.1f}")
print(f"  P25:     {scores_df['SCORE'].quantile(0.25):.1f}")
print(f"  P75:     {scores_df['SCORE'].quantile(0.75):.1f}")

## Analise de Decis e Lift

Tabela de decis com taxa de evento, lift acumulado e concentração de risco por faixa de score.

In [ ]:
# ---------------------------------------------------------------------------
# Tabela de decis
# ---------------------------------------------------------------------------
# Carregar target para análise de decis (apenas safras com target disponível)
fs_full = pd.read_parquet(
    LOCAL_DATA_PATH,
    columns=['NUM_CPF', 'SAFRA', TARGET]
)

scores_with_target = scores_df.merge(fs_full, on=['NUM_CPF', 'SAFRA'], how='left')
scores_valid = scores_with_target.dropna(subset=[TARGET])

if len(scores_valid) > 0:
    decile_df = decile_table(scores_valid[TARGET], scores_valid['PROBABILIDADE_FPD'])
    print("=== Tabela de Decis ===")
    print(decile_df.to_string(index=False))
else:
    print("AVISO: Target não disponível para análise de decis.")

In [ ]:
# ---------------------------------------------------------------------------
# Visualizações
# ---------------------------------------------------------------------------
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1) Histograma da distribuição de scores
axes[0].hist(scores_df['SCORE'], bins=50, color='#2196F3', edgecolor='black', alpha=0.8)
axes[0].set_title('Distribuição de Scores')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Frequência')

# 2) Gráfico de pizza — Faixas de risco
risk_colors = {'CRITICO': '#F44336', 'ALTO': '#FF9800', 'MEDIO': '#FFC107', 'BAIXO': '#4CAF50'}
pie_colors = [risk_colors[b] for b in band_order]
axes[1].pie(dist.values, labels=band_order, autopct='%1.1f%%', colors=pie_colors, startangle=90)
axes[1].set_title('Distribuição por Faixa de Risco')

# 3) Gráfico de lift por decil
if len(scores_valid) > 0 and 'lift' in decile_df.columns:
    axes[2].bar(range(1, len(decile_df) + 1), decile_df['lift'], color='#FF5722', edgecolor='black')
    axes[2].axhline(y=1.0, color='gray', linestyle='--')
    axes[2].set_title('Lift por Decil')
    axes[2].set_xlabel('Decil')
    axes[2].set_ylabel('Lift')
else:
    axes[2].text(0.5, 0.5, 'Lift não disponível\n(sem target)', ha='center', va='center', fontsize=14)
    axes[2].set_title('Lift por Decil')

plt.tight_layout()
plots_dir = os.path.join(ARTIFACT_DIR, 'plots')
os.makedirs(plots_dir, exist_ok=True)
plt.savefig(os.path.join(plots_dir, 'scoring_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Gráficos salvos em artifacts/plots/scoring_analysis.png")

## Escrita Particionada por SAFRA

Os scores são escritos no bucket Gold em formato Parquet, particionados por SAFRA para consultas eficientes.

In [ ]:
# ---------------------------------------------------------------------------
# Escrita particionada por SAFRA
# ---------------------------------------------------------------------------
# Escrita no bucket Gold (OCI Object Storage)
gold_scores_path = os.path.join(GOLD_FINAL_PATH, 'scores')
os.makedirs(gold_scores_path, exist_ok=True)

safras = sorted(scores_df['SAFRA'].unique())
for safra in safras:
    partition_df = scores_df[scores_df['SAFRA'] == safra]
    partition_path = os.path.join(gold_scores_path, f'SAFRA={safra}')
    os.makedirs(partition_path, exist_ok=True)
    output_file = os.path.join(partition_path, 'scores.parquet')
    partition_df.to_parquet(output_file, index=False)
    print(f"  SAFRA={safra}: {len(partition_df):,} registros -> {output_file}")

# Cópia local
local_scores_path = os.path.join(ARTIFACT_DIR, 'scores')
os.makedirs(local_scores_path, exist_ok=True)
scores_df.to_parquet(os.path.join(local_scores_path, 'scores_full.parquet'), index=False)

print(f"\nTotal: {len(scores_df):,} registros escritos em {len(safras)} partições")
print(f"Gold path: {gold_scores_path}")
print(f"Local path: {local_scores_path}")

log(f"08_scoring | {len(scores_df):,} registros scored | {len(safras)} SAFRAs | exec_id={execution_id}")